#  Data Chunking

Splits `streets.csv` (87,820,725 rows) from the landing volume into 4 chunks in the chunks volume, one per ingestion technique required by the project

| Chunk              | Format | Split | Rows      | Target Notebook      |
|--------------------|--------|-------|-----------|----------------------|
| streets_chunk_1.csv | CSV    | 50%   | 43,908,097   | COPY INTO            |
| streets_chunk_2.csv | CSV    | 20%   | 17,562,445   | DLT                  |
| streets_chunk_3.json| JSON   | 20%   | 17,562,445   | Auto Loader          |
| streets_chunk_4.xml | XML    | 10%   | 8,780,608   | PySpark XML          |


## Widgets & Configuration

In [0]:
import sys
import os

# Make src/utils importable from the notebook's working directory
NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in dir() else os.getcwd()
REPO_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from src.utils.chunk_io import write_single_file, reconcile_chunks

dbutils.widgets.text("catalog_name", "vstone_catalog", "1. Catalog Name")
dbutils.widgets.text("raw_schema", "raw", "2. Raw Schema")
dbutils.widgets.text("landing_volume", "landing", "3. Landing Volume")
dbutils.widgets.text("chunks_volume", "chunks", "4. Chunks Volume")
dbutils.widgets.text("split_seed", "42", "5. Random Split Seed")

CATALOG = dbutils.widgets.get("catalog_name")
RAW_SCHEMA = dbutils.widgets.get("raw_schema")
LANDING_VOL = dbutils.widgets.get("landing_volume")
CHUNKS_VOL = dbutils.widgets.get("chunks_volume")
SPLIT_SEED = int(dbutils.widgets.get("split_seed"))

LANDING_PATH = f"/Volumes/{CATALOG}/{RAW_SCHEMA}/{LANDING_VOL}"
CHUNKS_PATH = f"/Volumes/{CATALOG}/{RAW_SCHEMA}/{CHUNKS_VOL}"
SOURCE_CSV = f"{LANDING_PATH}/streets.csv"
SPLIT_WEIGHTS = [0.50, 0.20, 0.20, 0.10]  # must sum to 1.0 — see assumption #5

print(f"Landing path : {LANDING_PATH}")
print(f"Chunks path  : {CHUNKS_PATH}")
print(f"Source file  : {SOURCE_CSV}")
print(f"Split weights: {SPLIT_WEIGHTS} (seed={SPLIT_SEED})")


## Step 0 — Pre-flight: confirm source file exists

In [0]:
try:
    dbutils.fs.ls(SOURCE_CSV)
except Exception:
    raise FileNotFoundError(
        f"Source file not found: {SOURCE_CSV}\n"
        f"Upload streets.csv to the landing volume first (see README 'Data Setup')."
    )
print(f"Source file confirmed: {SOURCE_CSV}")

## Step 1 — Load source (explicit schema, no inference)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

# All columns loaded as STRING — bronze convention, no type inference.
# Typed casting happens in Silver, not here.
STREETS_SCHEMA = StructType([
    StructField("noise", StringType(), True),
    StructField("pollution", StringType(), True),
    StructField("date", StringType(), True),
    StructField("light", StringType(), True),
    StructField("raining", StringType(), True),
    StructField("street_id", StringType(), True),
])

df = (spark.read
      .option("header", "true")
      .option("inferSchema", "false")
      .option("encoding", "UTF-8")
      .schema(STREETS_SCHEMA)
      .csv(SOURCE_CSV))

total_rows = df.count()
print(f"Total source rows: {total_rows:,}")

## Step 2 — Split into 4 chunks (distributed, reproducible)

In [0]:
chunk1, chunk2, chunk3, chunk4 = df.randomSplit(SPLIT_WEIGHTS, seed=SPLIT_SEED)

chunk_counts_preview = {
    "chunk_1_csv": chunk1.count(),
    "chunk_2_csv": chunk2.count(),
    "chunk_3_json": chunk3.count(),
    "chunk_4_xml": chunk4.count(),
}
for name, cnt in chunk_counts_preview.items():
    pct = cnt / total_rows * 100
    print(f"  {name:<14} {cnt:>10,} rows  ({pct:5.2f}%)")

## Step 3 — Write chunk 1 & chunk 2 as single named CSV files

In [0]:
path_chunk1 = write_single_file(
    chunk1, dbutils, CHUNKS_PATH, "streets_chunk_1.csv", "csv", {"header": "true"}
)
print(f"Wrote: {path_chunk1}")

path_chunk2 = write_single_file(
    chunk2, dbutils, CHUNKS_PATH, "streets_chunk_2.csv", "csv", {"header": "true"}
)
print(f"Wrote: {path_chunk2}")

## Step 4 — Write chunk 3 as a single named JSON file


In [0]:
path_chunk3 = write_single_file(
    chunk3, dbutils, CHUNKS_PATH, "streets_chunk_3.json", "json"
)
print(f"Wrote: {path_chunk3}")

# Stage a copy in an isolated folder for Auto Loader (Day 3), matching the
# reference project's pattern — keeps Auto Loader's stream state from
# picking up ghost metadata from the rest of the chunks directory.
ISOLATED_SRC = f"{CHUNKS_PATH}/isolated_json_source"
dbutils.fs.mkdirs(ISOLATED_SRC)
dbutils.fs.cp(path_chunk3, f"{ISOLATED_SRC}/streets_chunk_3.json")
print(f"Staged for Auto Loader: {ISOLATED_SRC}/streets_chunk_3.json")

## Step 5 — Write chunk 4 as a single named XML file

In [0]:
# Uses Databricks' native `xml` format (built into DBR 14.3+, no external
# library required — see assumption #7). `rowTag="record"` matches what
# the Day 3 PySpark XML bronze notebook will read.

CHUNK4_TMP = f"{CHUNKS_PATH}/_tmp_streets_chunk_4.xml"
CHUNK4_FINAL = f"{CHUNKS_PATH}/streets_chunk_4.xml"

try:
    (chunk4.coalesce(1)
     .write.mode("overwrite")
     .format("xml")
     .option("rowTag", "record")
     .save(CHUNK4_TMP))

    part_files = [
        f.path for f in dbutils.fs.ls(CHUNK4_TMP)
        if f.name.startswith("part-") and f.name.endswith(".xml")
    ]
    if len(part_files) != 1:
        raise RuntimeError(f"Expected 1 XML part-file, found {len(part_files)}")

    dbutils.fs.mv(part_files[0], CHUNK4_FINAL)
    dbutils.fs.rm(CHUNK4_TMP, recurse=True)
    print(f"Wrote: {CHUNK4_FINAL}")

except Exception as e:
    # Fallback: native XML format unavailable in this workspace.
    # Convert only the 10% XML chunk (not the full file) via driver-side
    # xml.etree, matching the reference project's approach at a scale it
    # can actually handle.
    print(f"WARNING: native 'xml' format failed ({e}). Falling back to xml.etree.")
    import xml.etree.ElementTree as ET

    rows = chunk4.collect()  # ~10% of 87.8M rows (~8.8M) — driver-side fallback only, not
    # the default path. At this row count `collect()` needs a high-memory driver; the
    # native `format("xml")` path above should always be preferred where available.
    root = ET.Element("data")
    for r in rows:
        record = ET.SubElement(root, "record")
        for field in STREETS_SCHEMA.fieldNames():
            elem = ET.SubElement(record, field)
            elem.text = str(r[field]) if r[field] is not None else ""
    ET.indent(root, space="  ")

    local_tmp = "/tmp/streets_chunk_4.xml"
    ET.ElementTree(root).write(local_tmp, encoding="utf-8", xml_declaration=True)
    dbutils.fs.cp(f"file:{local_tmp}", CHUNK4_FINAL)
    print(f"Wrote (fallback): {CHUNK4_FINAL}")


## Step 6 — Reconciliation

In [0]:
# Confirms every source row landed in exactly one chunk. Not just printed —
# raises if it fails, so a broken run can't pass silently.

passed, summary = reconcile_chunks(total_rows, chunk_counts_preview)

print(f"\n{'='*60}")
print(f"  RECONCILIATION")
print(f"{'='*60}")
for k, v in summary.items():
    print(f"  {k:<16}: {v:,}" if isinstance(v, int) else f"  {k:<16}: {v}")
print(f"{'='*60}")

if not passed:
    raise ValueError(f"Chunk reconciliation FAILED: {summary}")
print("  Reconciliation PASSED — all source rows accounted for.")


## Step 7 — Confirm exactly 4 chunk files exist

In [0]:
remaining = sorted([
    f.name for f in dbutils.fs.ls(CHUNKS_PATH)
    if not f.isDir()
])
print(f"Files in chunks volume: {remaining}")

expected = {"streets_chunk_1.csv", "streets_chunk_2.csv", "streets_chunk_3.json", "streets_chunk_4.xml"}
if set(remaining) != expected:
    raise Exception(f"Expected exactly {expected}, found {set(remaining)}")

print(f"\n{'='*60}")
print(f"  CHUNKING COMPLETE — 4 files confirmed in {CHUNKS_PATH}")
print(f"{'='*60}")
